# 02 — Models

Implements AASIST from scratch. No code is copied from the reference repo — it is consulted only to verify layer shapes and config values.

**Architecture:**
1. `SincConv1d` — 70 learnable mel-scale bandpass filters
2. `ResBlock` — 1-D residual block with Feature Map Scaling (FMS)
3. `RawNet2Encoder` — SincConv → ResBlocks → GRU → 2-D feature map `[B, 128, 8, T]`
4. `GraphAttentionLayer` — single-head pairwise GAT
5. `HeterogeneousGraphLayer` — intra-type + cross-type attention with master node
6. `GraphPool` — top-k ratio pooling
7. `AASIST` — full model; AASIST-L config: `gat_dims=[24,32]`

In [1]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

print(f"PyTorch {torch.__version__}")

PyTorch 2.6.0+cu124


## SincConv Layer

Learnable sinc bandpass filter bank. Each filter is parameterised by a `low_freq` and a `band_freq` (both `nn.Parameter`), initialised at mel-scale spacings. A Hamming window suppresses spectral leakage.

In [2]:
class SincConv1d(nn.Module):
    """
    Learnable sinc bandpass filter bank.
    forward: [B, 1, 64600] -> [B, n_filters, T]
    """
    MIN_LOW_HZ  = 50.0
    MIN_BAND_HZ = 50.0

    @staticmethod
    def _to_mel(hz):  return 2595.0 * math.log10(1.0 + hz / 700.0)
    @staticmethod
    def _to_hz(mel):  return 700.0 * (10.0 ** (mel / 2595.0) - 1.0)

    def __init__(self, n_filters=70, kernel_size=1024, stride=16,
                 padding=512, sample_rate=16000):
        super().__init__()
        if kernel_size % 2 == 0:
            kernel_size += 1
        self.stride      = stride
        self.padding     = padding
        self.sample_rate = sample_rate

        # Mel-scale initialisation
        mel_min = self._to_mel(self.MIN_LOW_HZ)
        mel_max = self._to_mel(sample_rate / 2.0 - self.MIN_BAND_HZ)
        mel_pts = np.linspace(mel_min, mel_max, n_filters + 1)
        hz_pts  = np.array([self._to_hz(m) for m in mel_pts])

        self.low_freq  = nn.Parameter(torch.from_numpy(hz_pts[:-1]).float().unsqueeze(1))
        self.band_freq = nn.Parameter(torch.from_numpy(hz_pts[1:] - hz_pts[:-1]).float().unsqueeze(1))

        half = (kernel_size - 1) // 2
        n    = torch.arange(-half, half + 1).float()
        win  = 0.54 - 0.46 * torch.cos(2.0 * math.pi * (n + half) / (kernel_size - 1))
        self.register_buffer("window", win)
        self.register_buffer("n", n)

    def _build_filters(self):
        sr   = float(self.sample_rate)
        low  = self.MIN_LOW_HZ  / sr + torch.abs(self.low_freq)  / sr
        high = low + self.MIN_BAND_HZ / sr + torch.abs(self.band_freq) / sr
        high = torch.clamp(high, max=0.5)
        n    = self.n.unsqueeze(0)
        filt = (2.0 * high * torch.sinc(2.0 * high * n)
              - 2.0 * low  * torch.sinc(2.0 * low  * n)) * self.window
        return filt.unsqueeze(1)   # [n_filters, 1, kernel_size]

    def forward(self, x):
        return F.conv1d(x, self._build_filters(),
                        stride=self.stride, padding=self.padding)


sinc = SincConv1d()
x    = torch.randn(2, 1, 64600)
print(f"SincConv1d : {x.shape} -> {sinc(x).shape}")

SincConv1d : torch.Size([2, 1, 64600]) -> torch.Size([2, 70, 4038])


## ResBlock with FMS

1-D residual block. Feature Map Scaling (FMS) is a channel-wise squeeze-and-excite: global average pool → FC → sigmoid → multiply.

In [3]:
class ResBlock(nn.Module):
    """
    1-D residual block with FMS.
    When first=True the pre-activation BN is skipped.
    """

    def __init__(self, in_ch, out_ch, first=False):
        super().__init__()
        self.first = first
        if not first:
            self.bn1 = nn.BatchNorm1d(in_ch)
        self.lrelu  = nn.LeakyReLU(0.3, inplace=True)
        self.conv1  = nn.Conv1d(in_ch, out_ch, 3, padding=1, bias=False)
        self.bn2    = nn.BatchNorm1d(out_ch)
        self.conv2  = nn.Conv1d(out_ch, out_ch, 3, padding=1, bias=False)
        self.skip   = (nn.Conv1d(in_ch, out_ch, 1, bias=False)
                       if in_ch != out_ch else nn.Identity())
        self.fms_fc = nn.Linear(out_ch, out_ch)
        self.pool   = nn.MaxPool1d(3)

    def forward(self, x):
        out = x if self.first else self.lrelu(self.bn1(x))
        out = self.lrelu(self.bn2(self.conv1(out)))
        out = self.conv2(out) + self.skip(x)
        # FMS: global avg pool -> FC -> sigmoid -> scale channels
        scale = torch.sigmoid(self.fms_fc(out.mean(2))).unsqueeze(2)
        return self.pool(out * scale)


blk = ResBlock(70, 20, first=True)
t   = torch.randn(2, 70, 4038)
print(f"ResBlock(70,20) : {t.shape} -> {blk(t).shape}")

ResBlock(70,20) : torch.Size([2, 70, 4038]) -> torch.Size([2, 20, 1346])


## RawNet2 Encoder

Converts raw waveform `[B, 1, 64600]` → 2-D feature map `[B, 128, 8, T']`.  
GRU hidden dim 1024 = 128 × 8, so the reshape gives 8 pseudo-frequency bins.  
For 64 600-sample input, T' ≈ 5 after six MaxPool1d(3) stages.

In [4]:
class RawNet2Encoder(nn.Module):
    GRU_HIDDEN = 1024   # 1024 = 128 * 8  ->  F_bins = 8

    def __init__(self):
        super().__init__()
        self.sinc       = SincConv1d()
        self.bn0        = nn.BatchNorm1d(70)
        self.res1       = ResBlock(70,  20,  first=True)
        self.res2       = ResBlock(20,  20)
        self.res3       = ResBlock(20,  128)
        self.res4       = ResBlock(128, 128)
        self.res5       = ResBlock(128, 128)
        self.res6       = ResBlock(128, 128)
        self.bn_pre_gru = nn.BatchNorm1d(128)
        self.gru        = nn.GRU(128, self.GRU_HIDDEN, batch_first=True)

    def forward(self, x):
        x = self.bn0(torch.abs(self.sinc(x)))          # [B, 70, ~4038]
        x = self.res1(x)                               # [B, 20, ~1346]
        x = self.res2(x)                               # [B, 20, ~448]
        x = self.res3(x)                               # [B, 128, ~149]
        x = self.res4(x)                               # [B, 128, ~49]
        x = self.res5(x)                               # [B, 128, ~16]
        x = self.res6(x)                               # [B, 128, ~5]
        x = self.bn_pre_gru(x).permute(0, 2, 1)       # [B, T', 128]
        self.gru.flatten_parameters()
        x, _ = self.gru(x)                             # [B, T', 1024]
        B, T, H = x.shape
        return x.view(B, T, 128, H // 128).permute(0, 2, 3, 1)  # [B, 128, 8, T']


enc = RawNet2Encoder()
x   = torch.randn(2, 1, 64600)
print(f"RawNet2Encoder : {x.shape} -> {enc(x).shape}")

RawNet2Encoder : torch.Size([2, 1, 64600]) -> torch.Size([2, 128, 8, 5])


## Graph Attention Layer

Single-head pairwise GAT: nodes attend to each other via element-wise product, projection, and softmax.

In [5]:
class GraphAttentionLayer(nn.Module):
    """
    Pairwise single-head GAT.
    forward: [B, N, D_in] -> [B, N, D_out]
    """

    def __init__(self, in_dim, out_dim, temperature=1.0, dropout=0.2):
        super().__init__()
        self.temperature      = temperature
        self.att_proj         = nn.Linear(in_dim, out_dim)
        self.att_weight       = nn.Parameter(torch.empty(out_dim, 1))
        nn.init.xavier_normal_(self.att_weight)
        self.proj_with_att    = nn.Linear(in_dim, out_dim)
        self.proj_without_att = nn.Linear(in_dim, out_dim)
        self.bn               = nn.BatchNorm1d(out_dim)
        self.drop             = nn.Dropout(dropout)
        self.act              = nn.SELU(inplace=True)

    def _att_map(self, x):
        N  = x.size(1)
        xi = x.unsqueeze(2).expand(-1, -1, N, -1)
        e  = torch.tanh(self.att_proj(xi * xi.transpose(1, 2)))
        return F.softmax(torch.matmul(e, self.att_weight) / self.temperature, dim=2)

    def _apply_bn(self, x):
        s = x.shape
        return self.bn(x.view(-1, s[-1])).view(s)

    def forward(self, x):
        x   = self.drop(x)
        att = self._att_map(x)
        out = (self.proj_with_att(torch.matmul(att.squeeze(-1), x))
             + self.proj_without_att(x))
        return self.act(self._apply_bn(out))


gat   = GraphAttentionLayer(128, 24)
nodes = torch.randn(2, 8, 128)
print(f"GraphAttentionLayer : {nodes.shape} -> {gat(nodes).shape}")

GraphAttentionLayer : torch.Size([2, 8, 128]) -> torch.Size([2, 8, 24])


## Graph Pool

In [6]:
class GraphPool(nn.Module):
    """Retain the top-k fraction of nodes by learned importance score."""

    def __init__(self, ratio: float, in_dim: int, dropout: float = 0.3):
        super().__init__()
        self.ratio = ratio
        self.proj  = nn.Linear(in_dim, 1)
        self.drop  = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.sig   = nn.Sigmoid()

    def forward(self, h):
        scores = self.sig(self.proj(self.drop(h)))          # [B, N, 1]
        n_keep = max(int(h.size(1) * self.ratio), 1)
        _, idx = torch.topk(scores, n_keep, dim=1)          # [B, n_keep, 1]
        idx    = idx.expand(-1, -1, h.size(2))
        return torch.gather(h * scores, 1, idx)             # [B, n_keep, D]


pool = GraphPool(0.5, 24)
h    = torch.randn(2, 8, 24)
print(f"GraphPool(0.5) : {h.shape} -> {pool(h).shape}")

GraphPool(0.5) : torch.Size([2, 8, 24]) -> torch.Size([2, 4, 24])


## Heterogeneous Graph Layer (HS-GAL)

Processes two node types (spectral S and temporal T) jointly:
- Intra-type attention: S↔S and T↔T
- Cross-type attention: S↔T
- Master node aggregated from all nodes

In [7]:
class HeterogeneousGraphLayer(nn.Module):
    """
    forward: (x_s [B,Ns,D], x_t [B,Nt,D], master [B,1,D]|None)
          -> (x_s [B,Ns,D_out], x_t [B,Nt,D_out], master [B,1,D_out])
    """

    def __init__(self, in_dim, out_dim, temperature=2.0, dropout=0.2):
        super().__init__()
        self.temperature = temperature
        self.proj_s      = nn.Linear(in_dim, in_dim)
        self.proj_t      = nn.Linear(in_dim, in_dim)
        self.att_proj    = nn.Linear(in_dim, out_dim)
        for name in ("att_w_ss", "att_w_tt", "att_w_st", "att_w_m"):
            p = nn.Parameter(torch.empty(out_dim, 1))
            nn.init.xavier_normal_(p)
            setattr(self, name, p)
        self.att_proj_m         = nn.Linear(in_dim, out_dim)
        self.proj_with_att      = nn.Linear(in_dim, out_dim)
        self.proj_without_att   = nn.Linear(in_dim, out_dim)
        self.proj_m_with_att    = nn.Linear(in_dim, out_dim)
        self.proj_m_without_att = nn.Linear(in_dim, out_dim)
        self.bn   = nn.BatchNorm1d(out_dim)
        self.drop = nn.Dropout(dropout)
        self.act  = nn.SELU(inplace=True)

    def _pairwise(self, x):
        N  = x.size(1)
        xi = x.unsqueeze(2).expand(-1, -1, N, -1)
        return xi * xi.transpose(1, 2)

    def _build_att(self, x, Ns, Nt):
        e     = torch.tanh(self.att_proj(self._pairwise(x)))
        board = torch.zeros(*e.shape[:3], 1, device=x.device)
        board[:, :Ns, :Ns] = torch.matmul(e[:, :Ns, :Ns], self.att_w_ss)
        board[:, Ns:, Ns:] = torch.matmul(e[:, Ns:, Ns:], self.att_w_tt)
        board[:, :Ns, Ns:] = torch.matmul(e[:, :Ns, Ns:], self.att_w_st)
        board[:, Ns:, :Ns] = torch.matmul(e[:, Ns:, :Ns], self.att_w_st)
        return F.softmax(board / self.temperature, dim=2)

    def _update_master(self, x, master):
        att = F.softmax(
            torch.matmul(torch.tanh(self.att_proj_m(x * master)), self.att_w_m)
            / self.temperature, dim=1)
        return (self.proj_m_with_att(torch.matmul(att.transpose(1, 2), x))
              + self.proj_m_without_att(master))

    def _apply_bn(self, x):
        s = x.shape
        return self.bn(x.view(-1, s[-1])).view(s)

    def forward(self, x_s, x_t, master=None):
        Ns, Nt = x_s.size(1), x_t.size(1)
        x      = torch.cat([self.proj_s(x_s), self.proj_t(x_t)], dim=1)
        if master is None:
            master = x.mean(1, keepdim=True)
        x   = self.drop(x)
        att = self._build_att(x, Ns, Nt)
        m   = self._update_master(x, master)
        out = (self.proj_with_att(torch.matmul(att.squeeze(-1), x))
             + self.proj_without_att(x))
        out = self.act(self._apply_bn(out))
        return out[:, :Ns], out[:, Ns:], m


hg      = HeterogeneousGraphLayer(24, 32)
xs, xt  = torch.randn(2, 8, 24), torch.randn(2, 5, 24)
s2, t2, m2 = hg(xs, xt)
print(f"HeterogeneousGraphLayer : s{xs.shape} t{xt.shape} -> s{s2.shape} t{t2.shape} m{m2.shape}")

HeterogeneousGraphLayer : storch.Size([2, 8, 24]) ttorch.Size([2, 5, 24]) -> storch.Size([2, 8, 32]) ttorch.Size([2, 5, 32]) mtorch.Size([2, 1, 32])


## Graph Module

Extracts spectral and temporal node sets from the 2-D feature map, runs two parallel HtrgGAT inference paths, and returns a fixed-size readout vector.

In [8]:
class GraphModule(nn.Module):
    """forward: [B, C, F, T] -> [B, 5*gat_dim1]"""

    def __init__(self, in_dim=128, gat_dim0=24, gat_dim1=32,
                 pool_ratio_s=0.4, pool_ratio_t=0.5,
                 pool_ratio_hs=0.7,
                 temp0=2.0, temp1=100.0):
        super().__init__()
        self.gat_s    = GraphAttentionLayer(in_dim, gat_dim0, temperature=temp0)
        self.gat_t    = GraphAttentionLayer(in_dim, gat_dim0, temperature=temp0)
        self.pool_s   = GraphPool(pool_ratio_s, gat_dim0)
        self.pool_t   = GraphPool(pool_ratio_t, gat_dim0)
        # Two parallel HtrgGAT inference paths
        self.hg1a     = HeterogeneousGraphLayer(gat_dim0, gat_dim1, temperature=temp1)
        self.hg1b     = HeterogeneousGraphLayer(gat_dim1, gat_dim1, temperature=temp1)
        self.pool_hs1 = GraphPool(pool_ratio_hs, gat_dim1)
        self.pool_ht1 = GraphPool(pool_ratio_hs, gat_dim1)
        self.hg2a     = HeterogeneousGraphLayer(gat_dim0, gat_dim1, temperature=temp1)
        self.hg2b     = HeterogeneousGraphLayer(gat_dim1, gat_dim1, temperature=temp1)
        self.pool_hs2 = GraphPool(pool_ratio_hs, gat_dim1)
        self.pool_ht2 = GraphPool(pool_ratio_hs, gat_dim1)
        self.drop_way    = nn.Dropout(0.2)
        self.readout_dim = 5 * gat_dim1
        self.master1     = nn.Parameter(torch.randn(1, 1, gat_dim0))
        self.master2     = nn.Parameter(torch.randn(1, 1, gat_dim0))

    def forward(self, feat):
        # feat: [B, C, F, T]
        e_s, _ = feat.abs().max(3);  e_s = e_s.permute(0, 2, 1)  # [B, F, C]
        e_t, _ = feat.abs().max(2);  e_t = e_t.permute(0, 2, 1)  # [B, T, C]

        s  = self.pool_s(self.gat_s(e_s))
        t  = self.pool_t(self.gat_t(e_t))
        B  = feat.size(0)
        m1 = self.master1.expand(B, -1, -1)
        m2 = self.master2.expand(B, -1, -1)

        # Path 1
        s1, t1, m1 = self.hg1a(s, t, master=m1)
        s1 = self.pool_hs1(s1);  t1 = self.pool_ht1(t1)
        s1a, t1a, m1a = self.hg1b(s1, t1, master=m1)
        s1, t1, m1 = s1 + s1a, t1 + t1a, m1 + m1a

        # Path 2
        s2, t2, m2 = self.hg2a(s, t, master=m2)
        s2 = self.pool_hs2(s2);  t2 = self.pool_ht2(t2)
        s2a, t2a, m2a = self.hg2b(s2, t2, master=m2)
        s2, t2, m2 = s2 + s2a, t2 + t2a, m2 + m2a

        # Element-wise max across paths
        s_o = self.drop_way(torch.max(s1, s2))
        t_o = self.drop_way(torch.max(t1, t2))
        m_o = self.drop_way(torch.max(m1, m2))

        return torch.cat([
            t_o.abs().max(1)[0], t_o.mean(1),
            s_o.abs().max(1)[0], s_o.mean(1),
            m_o.squeeze(1)
        ], dim=1)


gm   = GraphModule()
feat = torch.randn(2, 128, 8, 5)
rv   = gm(feat)
print(f"GraphModule : {feat.shape} -> {rv.shape}  (readout_dim={gm.readout_dim})")

GraphModule : torch.Size([2, 128, 8, 5]) -> torch.Size([2, 160])  (readout_dim=160)


## AASIST Full Model

AASIST-L config: `gat_dims=[24, 32]`, `pool_ratios=[0.4, 0.5, 0.7, 0.5]`, `temperatures=[2.0, 2.0, 100.0, 100.0]`.

In [9]:
class AASIST(nn.Module):
    """
    Audio Anti-Spoofing using Integrated Spectral and Temporal Graph Attention.
    forward: [B, 1, 64600] -> logits [B, 2]
    """
    GAT_DIMS    = [24, 32]
    POOL_RATIOS = [0.4, 0.5, 0.7, 0.5]
    TEMPS       = [2.0, 2.0, 100.0, 100.0]

    def __init__(self):
        super().__init__()
        gd, pr, tp = self.GAT_DIMS, self.POOL_RATIOS, self.TEMPS
        self.encoder = RawNet2Encoder()
        self.graph   = GraphModule(
            in_dim=128,
            gat_dim0=gd[0], gat_dim1=gd[1],
            pool_ratio_s=pr[0], pool_ratio_t=pr[1],
            pool_ratio_hs=pr[2],
            temp0=tp[0], temp1=tp[2],
        )
        self.drop = nn.Dropout(0.5)
        self.out  = nn.Linear(self.graph.readout_dim, 2)

    def forward(self, x):
        feat   = self.encoder(x)        # [B, 128, 8, T']
        hidden = self.graph(feat)       # [B, readout_dim]
        return self.out(self.drop(hidden))  # [B, 2]


model = AASIST()
total = sum(p.numel() for p in model.parameters())
enc_p = sum(p.numel() for p in model.encoder.parameters())
grp_p = sum(p.numel() for p in model.graph.parameters())
print(f"Total parameters : {total:,}")
print(f"  Encoder        : {enc_p:,}")
print(f"  Graph module   : {grp_p:,}")

with torch.no_grad():
    logits = model(torch.randn(2, 1, 64600))
print(f"\nForward pass: [2, 1, 64600] -> logits {logits.shape}")
print(f"Sample logits: {logits[0].tolist()}")

Total parameters : 4,026,904
  Encoder        : 3,977,968
  Graph module   : 48,614

Forward pass: [2, 1, 64600] -> logits torch.Size([2, 2])
Sample logits: [1.1316770315170288, -0.6930698752403259]
